In [ ]:
import hashlib
import os
from pathlib import Path
from time import perf_counter

import boto3
import mlflow
import numpy as np
import polars as pl
import sentence_transformers
import torch
from botocore.client import BaseClient
from botocore.config import Config
from dotenv import load_dotenv
from huggingface_hub import model_info
from sentence_transformers import SentenceTransformer

In [ ]:
data_dir = "../data/processed/epo_english_v1"

In [ ]:
device = (
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)

In [ ]:
device

## Сборка датасета

Одна строка на публикацию. Все классификации сохраняем списком структур, IPC/CPC — отдельными уникальными списками. Вычисления ленивые до `collect()`. Здесь оставляем контроль связи 1:1 и заполненности текстов; отдельные проверки источников выполнены при EDA.


In [ ]:
patents_df = pl.scan_parquet(f"{data_dir}/patents.parquet")
texts_df = pl.scan_parquet(f"{data_dir}/patent_texts.parquet")
classifications_df = pl.scan_parquet(f"{data_dir}/classifications.parquet")

classification_fields = [
    name for name in classifications_df.collect_schema().names() if name != "publication_id"
]

In [ ]:
labels = classifications_df.group_by("publication_id").agg(
    pl.struct(classification_fields)
    .sort_by(["classification_system", "sequence"])
    .alias("classifications"),
    pl.col("classification_code")
    .filter(pl.col("classification_system") == "CPCI")
    .drop_nulls()
    .unique()
    .sort()
    .alias("cpc_codes"),
    pl.col("classification_code")
    .filter(pl.col("classification_system") == "IPCR")
    .drop_nulls()
    .unique()
    .sort()
    .alias("ipc_codes"),
)

dataset = (
    patents_df.join(texts_df, on="publication_id", how="left", validate="1:1")
    .join(labels, on="publication_id", how="left", validate="1:1")
    .with_columns(
        pl.concat_str(
            pl.col("title").str.strip_chars(),
            pl.col("abstract").str.strip_chars(),
            separator="\n",
        ).alias("text")
    )
)

schema = dataset.collect_schema()
dataset = dataset.with_columns(
    pl.col(name).fill_null(pl.lit([], dtype=schema[name]))
    for name in ("classifications", "cpc_codes", "ipc_codes")
).sort("publication_id")

In [ ]:
df = dataset.collect()

if df.filter(
    pl.col("title").is_null()
    | pl.col("abstract").is_null()
    | (pl.col("title").str.strip_chars() == "")
    | (pl.col("abstract").str.strip_chars() == "")
).height:
    raise ValueError("Есть публикации с пустым title или abstract")

output_path = f"{data_dir}/patents_search.parquet"
df.write_parquet(output_path, compression="zstd")
print(f"Сохранено публикаций: {df.height:,}")
print(f"Файл: {output_path}")

## MLflow: эксперимент с dense embeddings

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")

EXPERIMENT_NAME = "patent-semantic-search"
MODEL_ID = "BAAI/bge-m3"
MODEL_REVISION = "main"
DATASET_VERSION = "epo_english_v1"
MAX_SEQ_LENGTH = 1024
BATCH_SIZE = 8
SHARD_SIZE = 5000
S3_BUCKET = os.getenv("RUSTFS_BUCKET", "datasets")
DATASET_KEY = "epo/english/v1/processed/patents_search.parquet"
RAW_SOURCE_URI = "s3://datasets/epo/english/v1/epo_english_3g_20261006.xml"

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def create_s3_client() -> BaseClient:
    required = ("RUSTFS_ENDPOINT", "RUSTFS_ACCESS_KEY", "RUSTFS_SECRET_KEY")
    missing = [name for name in required if not os.getenv(name)]
    if missing:
        raise ValueError(f"No .env.rustfs variables: {', '.join(missing)}")
    return boto3.client(
        "s3",
        endpoint_url=os.environ["RUSTFS_ENDPOINT"],
        aws_access_key_id=os.environ["RUSTFS_ACCESS_KEY"],
        aws_secret_access_key=os.environ["RUSTFS_SECRET_KEY"],
        config=Config(
            signature_version="s3v4",
            s3={"addressing_style": "path"},
        ),
    )


def upload_parquet(client: BaseClient, path: Path, key: str) -> str:
    # SHA256 хранится в метаданных объекта; дополнительно проверяется размер.
    digest = sha256_file(path)
    client.upload_file(
        str(path),
        S3_BUCKET,
        key,
        ExtraArgs={
            "ContentType": "application/vnd.apache.parquet",
            "Metadata": {"sha256": digest},
        },
    )
    metadata = client.head_object(Bucket=S3_BUCKET, Key=key)
    if metadata["ContentLength"] != path.stat().st_size:
        raise RuntimeError(f"Размер загруженного файла не совпал: {key}")
    object_metadata = {name.lower(): value for name, value in metadata.get("Metadata", {}).items()}
    if object_metadata.get("sha256") != digest:
        raise RuntimeError(f"Не совпали метаданные SHA256: {key}")
    return f"s3://{S3_BUCKET}/{key}"

In [ ]:
tracking_uri = os.getenv("MLFLOW_TRACKING_URI")
if not tracking_uri:
    raise ValueError("Заполни MLFLOW_TRACKING_URI в локальном .env")
if mlflow.active_run() is not None:
    raise RuntimeError("Сначала заверши предыдущий MLflow Run")

mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment(EXPERIMENT_NAME)

dataset_path = Path(output_path).resolve()
# Читаем именно сохранённый файл: digest соответствует фактическому входу.
inputs = (
    pl.scan_parquet(dataset_path).select("publication_id", "text").sort("publication_id").collect()
)
if inputs.is_empty() or inputs["publication_id"].n_unique() != inputs.height:
    raise ValueError("Вход пуст или publication_id не уникальны")
if inputs.filter(
    pl.col("publication_id").is_null()
    | pl.col("text").is_null()
    | (pl.col("text").str.strip_chars() == "")
).height:
    raise ValueError("Во входе есть пустые ID или тексты")

dataset_sha256 = sha256_file(dataset_path)
resolved_revision = model_info(MODEL_ID, revision=MODEL_REVISION).sha
model = SentenceTransformer(MODEL_ID, revision=resolved_revision, device=device)
model.max_seq_length = MAX_SEQ_LENGTH
dimension = model.get_embedding_dimension()
s3_client = create_s3_client()

with mlflow.start_run(run_name="bge-m3-dense-v1") as run:
    run_id = run.info.run_id
    result_dir = PROJECT_ROOT / "data" / "embeddings" / run_id
    result_dir.mkdir(parents=True, exist_ok=False)
    dataset_source = upload_parquet(s3_client, dataset_path, DATASET_KEY)

    tracked_dataset = mlflow.data.from_pandas(
        inputs.to_pandas(),
        name=f"patents_search_{DATASET_VERSION}",
        source=dataset_source,
        digest=dataset_sha256[:32],
    )
    mlflow.log_input(
        tracked_dataset,
        context="embedding",
        tags={"dataset_version": DATASET_VERSION, "sha256": dataset_sha256},
    )
    mlflow.set_tags(
        {
            "run.type": "embedding",
            "dataset.version": DATASET_VERSION,
            "dataset.raw_source": RAW_SOURCE_URI,
            "embedding.mode": "dense",
        }
    )
    mlflow.log_params(
        {
            "model_id": MODEL_ID,
            "model_revision": resolved_revision,
            "dataset_sha256": dataset_sha256,
            "input_format": "title + newline + abstract; strip edges",
            "max_seq_length": MAX_SEQ_LENGTH,
            "batch_size": BATCH_SIZE,
            "shard_size": SHARD_SIZE,
            "device": device,
            "normalize_embeddings": True,
            "embedding_dimension": dimension,
            "torch_version": torch.__version__,
            "sentence_transformers_version": sentence_transformers.__version__,
        }
    )

    parts = []
    token_lengths = []
    encode_seconds = 0.0
    total_started = perf_counter()
    for part_number, start in enumerate(range(0, inputs.height, SHARD_SIZE)):
        shard = inputs.slice(start, SHARD_SIZE)
        text_batch = shard["text"].to_list()
        # Длина до усечения, включая специальные токены.
        lengths = model.tokenizer(
            text_batch,
            truncation=False,
            padding=False,
            return_length=True,
        )["length"]
        token_lengths.extend(lengths)

        started = perf_counter()
        vectors = model.encode(
            text_batch,
            batch_size=BATCH_SIZE,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=True,
        ).astype(np.float32)
        elapsed = perf_counter() - started
        encode_seconds += elapsed
        if vectors.shape != (shard.height, dimension) or not np.isfinite(vectors).all():
            raise RuntimeError("Некорректные embeddings")

        part_path = result_dir / f"part-{part_number:05d}.parquet"
        shard.select("publication_id").with_columns(
            pl.Series("embedding", vectors.tolist(), dtype=pl.List(pl.Float32)),
        ).write_parquet(part_path, compression="zstd")

        key = f"epo/english/v1/embeddings/{run_id}/{part_path.name}"
        remote_uri = upload_parquet(s3_client, part_path, key)
        parts.append(
            {
                "part": part_number,
                "rows": shard.height,
                "local_path": str(part_path),
                "s3_uri": remote_uri,
                "sha256": sha256_file(part_path),
            }
        )
        # После каждой части остаётся manifest, даже если следующая не завершится.
        manifest = {
            "run_id": run_id,
            "dataset_version": DATASET_VERSION,
            "dataset_source": dataset_source,
            "dataset_sha256": dataset_sha256,
            "raw_source": RAW_SOURCE_URI,
            "model_id": MODEL_ID,
            "model_revision": resolved_revision,
            "max_seq_length": MAX_SEQ_LENGTH,
            "normalize_embeddings": True,
            "embedding_dimension": dimension,
            "expected_rows": inputs.height,
            "completed_rows": sum(part["rows"] for part in parts),
            "complete": start + shard.height == inputs.height,
            "parts": parts,
        }
        mlflow.log_dict(manifest, "embedding_manifest.json")
        mlflow.log_metrics(
            {
                "completed_rows": manifest["completed_rows"],
                "part_documents_per_second": shard.height / elapsed,
            },
            step=part_number,
        )
        print(f"Сохранено {manifest['completed_rows']:,}/{inputs.height:,}")

    lengths_array = np.asarray(token_lengths)
    mlflow.log_metrics(
        {
            "publication_count": inputs.height,
            "encode_seconds": encode_seconds,
            "documents_per_second": inputs.height / encode_seconds,
            "total_seconds": perf_counter() - total_started,
            "truncated_document_count": int((lengths_array > MAX_SEQ_LENGTH).sum()),
            "truncated_document_fraction": float((lengths_array > MAX_SEQ_LENGTH).mean()),
            "token_length_p50": float(np.quantile(lengths_array, 0.50)),
            "token_length_p95": float(np.quantile(lengths_array, 0.95)),
            "token_length_max": int(lengths_array.max()),
        }
    )
    mlflow.log_dict(
        {"token_lengths": token_lengths},
        "diagnostics/token_lengths.json",
    )
    print(f"MLflow Run ID: {run_id}")
    print(f"Embeddings: {result_dir}")